#### Agentic RAG with Groq

This notebook starts from the original RAG workflow:

**laptops_info.txt → Load → Split → Embed → Chroma → Retrieve**

and evolves it into:

**User Query → Agent → Retrieval Tool → Chroma → Evidence → Groq → Grounded Answer**

#### Design rule

The original project's job is preserved:
- use `laptops_info.txt` as the knowledge source
- split the source into chunks
- create embeddings
- store vectors in Chroma
- retrieve the most relevant chunks
- answer questions from the retrieved knowledge


#### Install the project dependencies

Run this cell once inside the project's Python environment.

The project uses Agno for the agent, Groq for generation, Chroma for vector retrieval, LangChain components for document processing, and a local Hugging Face embedding model so Google API is not required.


In [ ]:
%pip install -U agno groq langchain langchain-community langchain-core langchain-text-splitters chromadb python-dotenv sentence-transformers

#### Load the Groq API key

Only the Groq API key is required for the model/agent layer.

Create a `.env` file in the same folder as this notebook:

```text
GROQ_API_KEY="your_groq_api_key"
```

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY is missing. Create a .env file and add your Groq API key."
    )

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


#### Import the document-processing components

In [2]:
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
print("Document-processing components imported successfully.")

C:\Users\pmgho\AppData\Local\Temp\ipykernel_19836\162895863.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Document-processing components imported successfully.


#### Load the original knowledge source

The project uses `laptops_info.txt` as its knowledge base, exactly as in the original RAG project.


In [3]:
DATA_FILE = Path("laptops_info.txt")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "laptops_info.txt was not found. Keep laptops_info.txt beside this notebook."
    )

loader = TextLoader(
    str(DATA_FILE),
    encoding="utf-8"
)

raw_docs = loader.load()

print("Knowledge source:", DATA_FILE.name)
print("Loaded documents:", len(raw_docs))

Knowledge source: laptops_info.txt
Loaded documents: 1


#### Split the knowledge source into chunks

- `chunk_size=500`
- `chunk_overlap=50`

In [10]:
text_splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(raw_docs)

print("Total number of chunks:", len(docs))

if docs:
    print("\nExample chunk:")
    print(docs[0].page_content)
    print("################################################################")


Total number of chunks: 5

Example chunk:
# Budget Laptops for AI/ML (India, 2024)

1. HP Victus 15
- Price: ₹78,990
- CPU: AMD Ryzen 5 5600H
- GPU: NVIDIA RTX 3050 (4GB)
- RAM: 16GB DDR4
- Storage: 512GB SSD
- Good for: TensorFlow, PyTorch, basic ML training
- Comments: Strong GPU, good thermals, solid AI performance
################################################################


#### Create local embeddings
- embeddings are generated locally
- Chroma still performs vector similarity search


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embedding_model.embed_query(
    "Which laptop is good for machine learning?"
)

print("Local embedding model initialized.")
print("Embedding dimension:", len(test_vector))
print("First 5 values:", test_vector[:5])


C:\Users\pmgho\AppData\Local\Temp\ipykernel_19836\2297011722.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Local embedding model initialized.
Embedding dimension: 384
First 5 values: [-0.018942687660455704, -0.0709509551525116, 0.014696487225592136, 0.019148088991642, 0.05514978617429733]


#### Create the Chroma vector store

This keeps the original RAG architecture:

**documents → embeddings → Chroma**

The vector database stores the semantic representation of the laptop knowledge.


In [12]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    collection_name="prajwal_laptop_knowledge"
)
print("Chroma vector store created successfully.")


Chroma vector store created successfully.


#### Create the similarity retriever

Similarity search with `k=2`.

That behavior is preserved so the retrieval stage still returns the two most relevant chunks.


In [13]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

print("Retriever ready.")

Retriever ready.


#### Verify the original RAG retrieval behavior

Before introducing an agent, we verify that the underlying RAG system itself works.

This is an important engineering step: first prove retrieval works, then give retrieval to the agent as a tool.


In [14]:
query = "Which laptop is best for machine learning under ₹80,000?"

retrieved_docs = retriever.invoke(query)

print("Retrieved chunks:", len(retrieved_docs))
print("\nTop Retrieved Chunks:")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)


Retrieved chunks: 2

Top Retrieved Chunks:

--- Chunk 1 ---
# Budget Laptops for AI/ML (India, 2024)

1. HP Victus 15
- Price: ₹78,990
- CPU: AMD Ryzen 5 5600H
- GPU: NVIDIA RTX 3050 (4GB)
- RAM: 16GB DDR4
- Storage: 512GB SSD
- Good for: TensorFlow, PyTorch, basic ML training
- Comments: Strong GPU, good thermals, solid AI performance

--- Chunk 2 ---
# Laptop Buying Tips for AI/ML (2024)
- Prefer 16GB RAM or more
- Look for NVIDIA GPUs like GTX 1650, RTX 3050 or better
- Avoid integrated graphics for training models
- SSD preferred for fast data access
- Ryzen 5, i5 H-series or better recommended


#### Agentic Layer

##### Import Agno and Groq

In [15]:
from agno.agent import Agent
from agno.models.groq import Groq
print("Agno and Groq imported successfully.")

Agno and Groq imported successfully.


#### Create the retrieval tool

The retriever is exposed as an agent tool.

This is the key transition from ordinary RAG to Agentic RAG:

**Retriever is no longer only called directly by the notebook — the Agent can call it as part of its workflow.**


In [16]:
def search_laptop_knowledge(query: str) -> str:
    """Search the laptop knowledge base and return the most relevant source chunks."""

    results = retriever.invoke(query)

    if not results:
        return "No relevant information was found in the laptop knowledge base."

    formatted_results = []

    for i, doc in enumerate(results, start=1):
        formatted_results.append(
            f"Source Chunk {i}:\n{doc.page_content}"
        )

    return "\n\n".join(formatted_results)

print("Retrieval tool created successfully.")

Retrieval tool created successfully.


#### Create the Groq Agent

The Agent uses Groq's `openai/gpt-oss-120b` model.

Its instructions enforce the most important property of this project:

**Laptop facts must come from the supplied knowledge base.**

The agent should search before answering knowledge-base questions and should not invent missing specifications.

In [17]:
agent = Agent(
    model=Groq(
        id="openai/gpt-oss-120b",
        api_key=GROQ_API_KEY
    ),
    tools=[search_laptop_knowledge],
    instructions=[
        "You are an Agentic RAG assistant for the supplied laptop knowledge base.",
        "For laptop-information questions, use the search_laptop_knowledge tool before answering.",
        "Use the retrieved source chunks as the factual source of truth.",
        "Do not invent laptop specifications, prices, benchmarks, availability, or features.",
        "If the knowledge base does not contain enough information, clearly say so.",
        "When comparing laptops, use only facts present in the retrieved source.",
        "Give a concise but useful explanation of the answer."
    ],
    markdown=True
)

print("Groq Agent initialized successfully.")

Groq Agent initialized successfully.


#### Test the Agentic RAG workflow

This is the first complete Agentic RAG test.

The expected flow is:

**Question → Agent → Retrieval Tool → Chroma → Evidence → Groq → Answer**

In [18]:
question = "Which laptop is best for machine learning under ₹80,000?"

agent.print_response(
    question,
    user_id="prajwal_gothkar",
    stream=True
)

Output()

In [19]:
question = "Which is better for AI workloads, HP Victus 15 or Lenovo IdeaPad Gaming 3?"

agent.print_response(
    question,
    user_id="prajwal_gothkar",
    stream=True
)

Output()

In [20]:
question = "I want a laptop for deep learning and parallel processing under ₹80,000. Which option from the knowledge base would you recommend and why?"

agent.print_response(
    question,
    user_id="prajwal_gothkar",
    stream=True
)

Output()

##### Create some questions to test the agent's ability to answer based on the knowledge base.

- question = "Which laptop has an RTX 3050 GPU?"
- question = "Which laptops have 16GB RAM?"
- question = "Which laptop is suitable for deep learning models?"
- question = "Which laptop is suitable for deep learning models?"
- question = "Which laptop is good for basic machine learning tasks?"
- question = "Which laptop is the most affordable option in the list?"
- question = "Compare HP Victus 15 and ASUS TUF Gaming F15 for AI and machine learning."
- question = "Which is better for AI workloads, HP Victus 15 or Lenovo IdeaPad Gaming 3?"

------ Like that questions.......

#### Interactive Agentic RAG

Now the system can accept a new question from the user.

The agent decides how to use its retrieval capability and produces the final grounded response through Groq.


In [21]:
user_question = input("\nAsk your laptop knowledge question: ")

agent.print_response(
    user_question,
    user_id="prajwal_gothkar",
    stream=True
)

Output()

In [22]:
user_question = input("\nAsk your laptop knowledge question: ")

agent.print_response(
    user_question,
    user_id="prajwal_gothkar",
    stream=True
)

Output()

#### Retrieval Inspection Mode

##### Inspect the evidence used by the retrieval layer

This cell is intentionally kept separate from the Agent.

It allows the developer to inspect exactly what Chroma retrieves for a question.


In [23]:
inspection_question = "Which laptops have 16GB RAM and an NVIDIA GPU?"
evidence = search_laptop_knowledge(inspection_question)
print("Retrieved Evidence:")
print(evidence)

Retrieved Evidence:
Source Chunk 1:
2. Lenovo IdeaPad Gaming 3
- Price: ₹74,999
- CPU: Intel i5 11300H
- GPU: NVIDIA GTX 1650 (4GB)
- RAM: 8GB (expandable to 16GB)
- Storage: 512GB SSD
- Good for: Lightweight ML tasks, student projects
- Comments: Affordable gaming laptop, decent for AI workloads

Source Chunk 2:
5. Dell Inspiron 15
- Price: ₹69,500
- CPU: Intel i5 1235U
- GPU: Integrated (Iris Xe)
- RAM: 16GB
- Storage: 512GB SSD
- Good for: Basic ML, tabular data, sklearn models
- Comments: No dedicated GPU, not suitable for heavy AI work

6. MSI GF63 Thin
- Price: ₹76,990
- CPU: Intel i5 11400H
- GPU: NVIDIA GTX 1650
- RAM: 8GB
- Storage: 512GB SSD
- Good for: Basic DL models, CNNs
- Comments: Slim and lightweight, but limited GPU


# Agentic RAG Mental Model

The completed project has two clearly separated responsibilities.

#### Retrieval layer

```text
laptops_info.txt
      ↓
TextLoader
      ↓
CharacterTextSplitter
      ↓
Local Embeddings
      ↓
Chroma
      ↓
Similarity Retriever
```

### Agent layer

```text
User Question
      ↓
Groq Agent
      ↓
Retrieval Tool
      ↓
Chroma Retriever
      ↓
Evidence
      ↓
Groq GPT-OSS 120B
      ↓
Grounded Answer
```


#### Final Validation

##### Run a final end-to-end test

In [25]:
print("==============================================")
print(" PRAJWAL AGENTIC RAG VALIDATION")
print("==============================================")
print("- Knowledge file loaded")
print("- Documents chunked")
print("- Local embeddings initialized")
print("- Chroma vector store initialized")
print("- Similarity retriever initialized")
print("- Groq model configured")
print("- Agent initialized")
print("- Retrieval tool connected")
print("----------------------------------------------")
print("Agentic RAG pipeline is ready.")

 PRAJWAL AGENTIC RAG VALIDATION
- Knowledge file loaded
- Documents chunked
- Local embeddings initialized
- Chroma vector store initialized
- Similarity retriever initialized
- Groq model configured
- Agent initialized
- Retrieval tool connected
----------------------------------------------
Agentic RAG pipeline is ready.
